<a href="https://colab.research.google.com/github/aszczi/Urban_mobility_in_Cracow/blob/main/Mobilnosc_miejska_w_Krakowie_JOIN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Wgranie pliku z juz zebranymi danymi

In [ ]:
import os
import requests
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files
from google.colab import output
output.enable_custom_widget_manager()

GITHUB_CSV_URL = "https://raw.githubusercontent.com/aszczi/Urban_mobility_in_Cracow/refs/heads/main/traffic_history.csv"
CSV_FILE = "traffic_history.csv"
data_ready = False

status_out = widgets.Output()
github_btn = widgets.Button(description="Pobierz z GitHub", button_style='info', icon='download')
upload_btn = widgets.Button(description="Wgraj z dysku", button_style='info', icon='upload')
ok_btn = widgets.Button(description="Zatwierdź (OK)", button_style='success', disabled=True, icon='check')

def mark_ready():
    global data_ready
    data_ready = True
    ok_btn.disabled = False
    with status_out:
        print("Dane są gotowe do zatwierdzenia.")

def on_github_clicked(_):
    with status_out:
        clear_output()
        print("Pobieranie z GitHub...")
        try:
            res = requests.get(GITHUB_CSV_URL, timeout=20)
            res.raise_for_status()
            with open(CSV_FILE, 'wb') as f: f.write(res.content)
            print(f"Pobrano: {CSV_FILE} ({os.path.getsize(CSV_FILE)} bajtów)")
            mark_ready()
        except Exception as e:
            print(f"Błąd pobierania: {e}")

def on_upload_clicked(_):
    with status_out:
        clear_output()
        print("Oczekiwanie na wybór pliku...")
        uploaded = files.upload()
        if uploaded:
            fname = list(uploaded.keys())[0]
            with open(CSV_FILE, "wb") as f: f.write(uploaded[fname])
            print(f"Wgrano: {fname} i zapisano jako {CSV_FILE}")
            mark_ready()
        else:
            print("Nie wybrano pliku.")

def on_ok_clicked(_):
    github_btn.disabled = True
    upload_btn.disabled = True
    ok_btn.disabled = True
    with status_out:
        clear_output()
        print("Dane zatwierdzone. Możesz przejść do kolejnych komórek.")

github_btn.on_click(on_github_clicked)
upload_btn.on_click(on_upload_clicked)
ok_btn.on_click(on_ok_clicked)

print("KONFIGURACJA DANYCH")
display(widgets.HBox([github_btn, upload_btn]))
display(ok_btn)
display(status_out)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

check_out = widgets.Output()
check_btn = widgets.Button(description="Sprawdź i Odblokuj", button_style='warning')

def on_check_clicked(_):
    with check_out:
        clear_output()
        if data_ready:
            print("Dane gotowe. Możesz teraz uruchomić kolejne komórki.")
            check_btn.button_style = 'success'
        else:
            print("Najpierw wybierz źródło i kliknij 'Zatwierdź (OK)' w pierwszej komórce!")

check_btn.on_click(on_check_clicked)
display(check_btn, check_out)

if not data_ready:
    raise RuntimeError("ZATRZYMANO: Dane nie zostały jeszcze zatwierdzone w pierwszej komórce. Kliknij OK powyżej i spróbuj ponownie.")
else:
    print("Weryfikacja pozytywna. Kontynuacja...")

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime
from google.colab import userdata
import time
import os

TOMTOM_API_KEY = 'ZW0nhcWdTcUE6SJjSJV5f9vXyLKbBHuR'

if not TOMTOM_API_KEY:
    raise ValueError(
        "Nie udało się pobrać klucza TOMTOM_API_KEY"
    )

CSV_FILE = "traffic_history.csv"


POINTS = [
    {
        "name": "Rondo Mogilskie",
        "lat": 50.0672,
        "lon": 19.9595
    },
    {
        "name": "Lubomirskiego",
        "lat": 50.0676,
        "lon": 19.9536
    },
    {
        "name": "Aleja Powstania Warszawskiego",
        "lat": 50.0647,
        "lon": 19.9608
    },
    {
        "name": "Grzegórzecka",
        "lat": 50.0578,
        "lon": 19.9564
    },
    {
        "name": "Rondo Grzegórzeckie",
        "lat": 50.0576,
        "lon": 19.9609
    },
    {
        "name": "Galeria Krakowska / Pawia",
        "lat": 50.0669,
        "lon": 19.9457
    },
    {
        "name": "Dworzec Główny Tunel",
        "lat": 50.0686,
        "lon": 19.9477
    },
    {
        "name": "Aleje Trzech Wieszczów - AGH",
        "lat": 50.0674,
        "lon": 19.9209
    },
    {
        "name": "Plac Inwalidów",
        "lat": 50.0671,
        "lon": 19.9269
    },
    {
        "name": "Nowy Kleparz",
        "lat": 50.0731,
        "lon": 19.9352
    },
    {
        "name": "Aleja Słowackiego / Łobzowska",
        "lat": 50.0692,
        "lon": 19.9302
    },
    {
        "name": "Cracovia / Aleja Mickiewicza",
        "lat": 50.0595,
        "lon": 19.9235
    },
    {
        "name": "Rondo Grunwaldzkie",
        "lat": 50.0492,
        "lon": 19.9317
    },
    {
        "name": "Most Dębnicki",
        "lat": 50.0530,
        "lon": 19.9278
    },
    {
        "name": "Rondo Matecznego",
        "lat": 50.0364,
        "lon": 19.9406
    },
    {
        "name": "Kalwaryjska",
        "lat": 50.0412,
        "lon": 19.9443
    },
    {
        "name": "Bonarka / Kamieńskiego",
        "lat": 50.0298,
        "lon": 19.9513
    },
    {
        "name": "Zakopiańska / Brożka",
        "lat": 50.0227,
        "lon": 19.9349
    },
    {
        "name": "Łagiewniki",
        "lat": 50.0202,
        "lon": 19.9378
    },
    {
        "name": "Opolska Estakada",
        "lat": 50.0857,
        "lon": 19.9536
    },
    {
        "name": "Rondo Polsadu",
        "lat": 50.0879,
        "lon": 19.9618
    },
    {
        "name": "Rondo Barei",
        "lat": 50.0911,
        "lon": 19.9742
    },
    {
        "name": "Aleja 29 Listopada / Opolska",
        "lat": 50.0861,
        "lon": 19.9520
    },
    {
        "name": "Aleja 29 Listopada / Dobrego Pasterza",
        "lat": 50.0953,
        "lon": 19.9746
    },
    {
        "name": "Rondo Ofiar Katynia",
        "lat": 50.0879,
        "lon": 19.8976
    },
    {
        "name": "Bronowice SKA",
        "lat": 50.0814,
        "lon": 19.8994
    },
    {
        "name": "Armii Krajowej / Zarzecze",
        "lat": 50.0738,
        "lon": 19.8985
    },
    {
        "name": "Czarnowiejska / Armii Krajowej",
        "lat": 50.0690,
        "lon": 19.9075
    },
    {
        "name": "Rondo Czyżyńskie",
        "lat": 50.0729,
        "lon": 20.0167
    },
    {
        "name": "Plac Centralny",
        "lat": 50.0720,
        "lon": 20.0372
    },
    {
        "name": "Aleja Pokoju / Centralna",
        "lat": 50.0667,
        "lon": 20.0028
    },
    {
        "name": "Nowohucka / Klimeckiego",
        "lat": 50.0503,
        "lon": 19.9765
    },
    {
        "name": "M1 / Aleja Pokoju",
        "lat": 50.0661,
        "lon": 20.0149
    },
    {
        "name": "Wielicka / Powstańców Wielkopolskich",
        "lat": 50.0335,
        "lon": 19.9621
    },
    {
        "name": "Estakada Obrońców Lwowa",
        "lat": 50.0396,
        "lon": 19.9627
    },
    {
        "name": "Bieżanowska / Wielicka",
        "lat": 50.0207,
        "lon": 19.9820
    },
    {
        "name": "Prokocim Szpital",
        "lat": 50.0117,
        "lon": 20.0005
    },
    {
        "name": "Węzeł Łagiewniki",
        "lat": 50.0136,
        "lon": 19.9325
    },
    {
        "name": "Węzeł Balice / A4",
        "lat": 50.0878,
        "lon": 19.7937
    },
    {
        "name": "Węzeł Tyniec / A4",
        "lat": 50.0196,
        "lon": 19.8078
    },
    {
        "name": "Węzeł Wielicka / A4",
        "lat": 50.0034,
        "lon": 20.0065
    }
]


def fetch_tomtom_flow(api_key, lat, lon):
    api_key = api_key.strip()

    if not api_key:
        raise ValueError("Brak klucza TOMTOM_API_KEY.")

    url = "https://api.tomtom.com/traffic/services/4/flowSegmentData/absolute/18/json"

    params = {
        "key": api_key,
        "point": f"{lat},{lon}",
        "unit": "kmph",
        "openLr": "false"
    }

    response = requests.get(url, params=params, timeout=15)

    if not response.ok:
        raise RuntimeError(
            f"TomTom HTTP {response.status_code}: {response.text[:300]}"
        )

    data = response.json()
    flow = data.get("flowSegmentData")

    if not flow:
        raise RuntimeError(f"Brak pola flowSegmentData w odpowiedzi: {data}")

    return flow


def collect_once():
    rows = []
    now = datetime.now()

    for point in POINTS:
        try:
            flow = fetch_tomtom_flow(
                TOMTOM_API_KEY,
                point["lat"],
                point["lon"]
            )

            current_speed = flow.get("currentSpeed")
            free_flow_speed = flow.get("freeFlowSpeed")
            current_travel_time = flow.get("currentTravelTime")
            free_flow_travel_time = flow.get("freeFlowTravelTime")
            confidence = flow.get("confidence")
            road_closure = flow.get("roadClosure")

            if current_travel_time is not None and free_flow_travel_time is not None:
                delay_sec = max(0, current_travel_time - free_flow_travel_time)
            else:
                delay_sec = None

            if current_speed is not None and free_flow_speed not in [None, 0]:
                speed_ratio = current_speed / free_flow_speed
                congestion_index = round((1 - speed_ratio) * 100, 2)
                congestion_index = max(0, congestion_index)
            else:
                speed_ratio = None
                congestion_index = None

            rows.append({
                "timestamp": now,
                "point_name": point["name"],
                "lat": point["lat"],
                "lon": point["lon"],
                "current_speed_kmph": current_speed,
                "free_flow_speed_kmph": free_flow_speed,
                "current_travel_time_sec": current_travel_time,
                "free_flow_travel_time_sec": free_flow_travel_time,
                "delay_sec": delay_sec,
                "speed_ratio": speed_ratio,
                "congestion_index_pct": congestion_index,
                "confidence": confidence,
                "road_closure": road_closure
            })

            print(
                f"{point['name']}: "
                f"{current_speed} km/h, "
                f"opóźnienie {delay_sec} s, "
                f"indeks korka {congestion_index}%"
            )

        except Exception as e:
            print(f"Błąd dla punktu {point['name']}: {e}")

    return pd.DataFrame(rows)


def save_to_csv(df):
    file_exists = os.path.exists(CSV_FILE)

    df.to_csv(
        CSV_FILE,
        mode="a",
        header=not file_exists,
        index=False,
        encoding="utf-8"
    )

    print(f"Zapisano {len(df)} rekordów do {CSV_FILE}")


def collect_loop(interval_minutes=5, number_of_rounds=12):
    for i in range(number_of_rounds):
        print(f"\nPomiar {i + 1}/{number_of_rounds}")
        df = collect_once()

        if not df.empty:
            save_to_csv(df)

        if i < number_of_rounds - 1:
            time.sleep(interval_minutes * 60)


def load_history():
    if not os.path.exists(CSV_FILE):
        raise FileNotFoundError(
            f"Nie ma pliku {CSV_FILE}. Najpierw uruchom collect_once() albo collect_loop()."
        )

    df = pd.read_csv(CSV_FILE)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df["hour"] = df["timestamp"].dt.hour
    df["date"] = df["timestamp"].dt.date

    return df


def show_summary(top_n=None):
    df = load_history()

    summary = (
        df.groupby("point_name")
        .agg(
            liczba_pomiarow=("timestamp", "count"),
            srednia_predkosc_kmh=("current_speed_kmph", "mean"),
            min_predkosc_kmh=("current_speed_kmph", "min"),
            max_predkosc_kmh=("current_speed_kmph", "max"),
            srednie_opoznienie_sec=("delay_sec", "mean"),
            max_opoznienie_sec=("delay_sec", "max"),
            sredni_indeks_korka_pct=("congestion_index_pct", "mean")
        )
        .round(2)
        .reset_index()
        .sort_values("srednie_opoznienie_sec", ascending=False)
    )

    if top_n:
        summary = summary.head(top_n)

    print(summary.to_string(index=False))


def show_average_speed_summary(top_n=30):
    df = load_history()

    summary = (
        df.groupby("point_name")
        .agg(
            srednia_predkosc_kmh=("current_speed_kmph", "mean"),
            minimalna_predkosc_kmh=("current_speed_kmph", "min"),
            maksymalna_predkosc_kmh=("current_speed_kmph", "max"),
            srednie_opoznienie_sec=("delay_sec", "mean"),
            liczba_pomiarow=("timestamp", "count")
        )
        .round(2)
        .reset_index()
        .sort_values("srednia_predkosc_kmh", ascending=True)
        .head(top_n)
    )

    print(summary.to_string(index=False))


def plot_current_congestion_ranking(top_n=20):
    df = load_history()

    summary = (
        df.groupby("point_name")
        .agg(
            avg_delay_sec=("delay_sec", "mean"),
            max_delay_sec=("delay_sec", "max"),
            avg_speed=("current_speed_kmph", "mean"),
            avg_congestion=("congestion_index_pct", "mean"),
            measurements=("timestamp", "count")
        )
        .reset_index()
        .sort_values("avg_delay_sec", ascending=False)
        .head(top_n)
    )

    plt.figure(figsize=(12, max(6, top_n * 0.35)))
    plt.barh(summary["point_name"], summary["avg_delay_sec"])

    plt.gca().invert_yaxis()
    plt.title(f"TOP {top_n} najbardziej zakorkowanych punktów")
    plt.xlabel("Średnie opóźnienie [s]")
    plt.ylabel("Lokalizacja")
    plt.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_speed_ranking(top_n=20):
    df = load_history()

    summary = (
        df.groupby("point_name")
        .agg(
            avg_speed=("current_speed_kmph", "mean"),
            avg_delay_sec=("delay_sec", "mean"),
            avg_congestion=("congestion_index_pct", "mean"),
            measurements=("timestamp", "count")
        )
        .reset_index()
        .sort_values("avg_speed", ascending=True)
        .head(top_n)
    )

    plt.figure(figsize=(12, max(6, top_n * 0.35)))
    plt.barh(summary["point_name"], summary["avg_speed"])

    plt.gca().invert_yaxis()
    plt.title(f"TOP {top_n} punktów z najniższą średnią prędkością")
    plt.xlabel("Średnia prędkość [km/h]")
    plt.ylabel("Lokalizacja")
    plt.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_average_speed_by_point(top_n=25):
    df = load_history()

    summary = (
        df.groupby("point_name")["current_speed_kmph"]
        .mean()
        .sort_values(ascending=True)
        .head(top_n)
    )

    plt.figure(figsize=(12, max(6, top_n * 0.35)))
    plt.barh(summary.index, summary.values)

    plt.gca().invert_yaxis()
    plt.title(f"Średnia prędkość — TOP {top_n} najwolniejszych lokalizacji")
    plt.xlabel("Średnia prędkość [km/h]")
    plt.ylabel("Lokalizacja")
    plt.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_latest_snapshot(top_n=20):
    df = load_history()

    latest_time = df["timestamp"].max()
    latest = df[df["timestamp"] == latest_time].copy()

    latest = latest.sort_values("delay_sec", ascending=False).head(top_n)

    plt.figure(figsize=(12, max(6, top_n * 0.35)))
    plt.barh(latest["point_name"], latest["delay_sec"])

    plt.gca().invert_yaxis()
    plt.title(f"Najnowszy pomiar korków — {latest_time}")
    plt.xlabel("Opóźnienie [s]")
    plt.ylabel("Lokalizacja")
    plt.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_latest_speed_snapshot(top_n=20):
    df = load_history()

    latest_time = df["timestamp"].max()
    latest = df[df["timestamp"] == latest_time].copy()

    latest = latest.sort_values("current_speed_kmph", ascending=True).head(top_n)

    plt.figure(figsize=(12, max(6, top_n * 0.35)))
    plt.barh(latest["point_name"], latest["current_speed_kmph"])

    plt.gca().invert_yaxis()
    plt.title(f"Najnowszy pomiar prędkości — {latest_time}")
    plt.xlabel("Prędkość [km/h]")
    plt.ylabel("Lokalizacja")
    plt.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_delay_by_hour_top(top_n=8):
    df = load_history()

    top_points = (
        df.groupby("point_name")["delay_sec"]
        .mean()
        .sort_values(ascending=False)
        .head(top_n)
        .index
    )

    filtered = df[df["point_name"].isin(top_points)]

    hourly = (
        filtered.groupby(["hour", "point_name"])["delay_sec"]
        .mean()
        .reset_index()
    )

    plt.figure(figsize=(15, 7))

    for point_name in top_points:
        subset = hourly[hourly["point_name"] == point_name]
        plt.plot(
            subset["hour"],
            subset["delay_sec"],
            marker="o",
            linewidth=2,
            label=point_name
        )

    plt.title(f"Średnie opóźnienie według godziny — TOP {top_n} lokalizacji")
    plt.xlabel("Godzina")
    plt.ylabel("Średnie opóźnienie [s]")
    plt.xticks(range(0, 24))
    plt.grid(True, alpha=0.3)

    plt.legend(
        title="Lokalizacja",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        borderaxespad=0
    )

    plt.tight_layout()
    plt.show()


def plot_congestion_by_hour_top(top_n=8):
    df = load_history()

    top_points = (
        df.groupby("point_name")["congestion_index_pct"]
        .mean()
        .sort_values(ascending=False)
        .head(top_n)
        .index
    )

    filtered = df[df["point_name"].isin(top_points)]

    hourly = (
        filtered.groupby(["hour", "point_name"])["congestion_index_pct"]
        .mean()
        .reset_index()
    )

    plt.figure(figsize=(15, 7))

    for point_name in top_points:
        subset = hourly[hourly["point_name"] == point_name]
        plt.plot(
            subset["hour"],
            subset["congestion_index_pct"],
            marker="o",
            linewidth=2,
            label=point_name
        )

    plt.title(f"Indeks zakorkowania według godziny — TOP {top_n} lokalizacji")
    plt.xlabel("Godzina")
    plt.ylabel("Indeks zakorkowania [%]")
    plt.xticks(range(0, 24))
    plt.grid(True, alpha=0.3)

    plt.legend(
        title="Lokalizacja",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        borderaxespad=0
    )

    plt.tight_layout()
    plt.show()


def plot_average_speed_by_hour(top_n=8):
    df = load_history()

    top_points = (
        df.groupby("point_name")["current_speed_kmph"]
        .mean()
        .sort_values(ascending=True)
        .head(top_n)
        .index
    )

    filtered = df[df["point_name"].isin(top_points)]

    hourly = (
        filtered.groupby(["hour", "point_name"])["current_speed_kmph"]
        .mean()
        .reset_index()
    )

    plt.figure(figsize=(15, 7))

    for point_name in top_points:
        subset = hourly[hourly["point_name"] == point_name]
        plt.plot(
            subset["hour"],
            subset["current_speed_kmph"],
            marker="o",
            linewidth=2,
            label=point_name
        )

    plt.title(f"Średnia prędkość według godziny — TOP {top_n} najwolniejszych lokalizacji")
    plt.xlabel("Godzina")
    plt.ylabel("Średnia prędkość [km/h]")
    plt.xticks(range(0, 24))
    plt.grid(True, alpha=0.3)

    plt.legend(
        title="Lokalizacja",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        borderaxespad=0
    )

    plt.tight_layout()
    plt.show()



def plot_delay_heatmap(top_n=25):
    df = load_history()

    top_points = (
        df.groupby("point_name")["delay_sec"]
        .mean()
        .sort_values(ascending=False)
        .head(top_n)
        .index
    )

    filtered = df[df["point_name"].isin(top_points)]

    pivot = filtered.pivot_table(
        index="point_name",
        columns="hour",
        values="delay_sec",
        aggfunc="mean"
    )

    pivot["avg"] = pivot.mean(axis=1)
    pivot = pivot.sort_values("avg", ascending=False).drop(columns="avg")

    plt.figure(figsize=(16, max(7, top_n * 0.35)))
    plt.imshow(
        pivot,
        aspect="auto",
        interpolation="nearest"
    )

    plt.colorbar(label="Średnie opóźnienie [s]")
    plt.title(f"Heatmapa opóźnień — TOP {top_n} lokalizacji")
    plt.xlabel("Godzina")
    plt.ylabel("Lokalizacja")

    plt.xticks(
        ticks=range(len(pivot.columns)),
        labels=pivot.columns
    )

    plt.yticks(
        ticks=range(len(pivot.index)),
        labels=pivot.index
    )

    plt.tight_layout()
    plt.show()


def plot_speed_heatmap(top_n=25):
    df = load_history()

    top_points = (
        df.groupby("point_name")["current_speed_kmph"]
        .mean()
        .sort_values(ascending=True)
        .head(top_n)
        .index
    )

    filtered = df[df["point_name"].isin(top_points)]

    pivot = filtered.pivot_table(
        index="point_name",
        columns="hour",
        values="current_speed_kmph",
        aggfunc="mean"
    )

    pivot["avg"] = pivot.mean(axis=1)
    pivot = pivot.sort_values("avg", ascending=True).drop(columns="avg")

    plt.figure(figsize=(16, max(7, top_n * 0.35)))
    plt.imshow(
        pivot,
        aspect="auto",
        interpolation="nearest"
    )

    plt.colorbar(label="Średnia prędkość [km/h]")
    plt.title(f"Heatmapa prędkości — TOP {top_n} najwolniejszych lokalizacji")
    plt.xlabel("Godzina")
    plt.ylabel("Lokalizacja")

    plt.xticks(
        ticks=range(len(pivot.columns)),
        labels=pivot.columns
    )

    plt.yticks(
        ticks=range(len(pivot.index)),
        labels=pivot.index
    )

    plt.tight_layout()
    plt.show()


def plot_interactive_map():
    """Rysuje interaktywną mapę Krakowa z najnowszymi opóźnieniami"""
    df = load_history()

    latest_time = df["timestamp"].max()
    latest = df[df["timestamp"] == latest_time].copy()


    latest["size_for_map"] = latest["delay_sec"].apply(lambda x: max(x, 1))

    fig = px.scatter_mapbox(
        latest,
        lat="lat",
        lon="lon",
        color="delay_sec",
        size="size_for_map",
        hover_name="point_name",
        hover_data={
            "delay_sec": True,
            "current_speed_kmph": True,
            "congestion_index_pct": True,
            "lat": False,
            "lon": False,
            "size_for_map": False
        },
        color_continuous_scale=px.colors.sequential.YlOrRd,
        zoom=11.5,
        center={"lat": 50.0614, "lon": 19.9383},
        title=f"Interaktywna mapa opóźnień w Krakowie (Pomiar: {latest_time})",
        labels={
            "delay_sec": "Opóźnienie [s]",
            "current_speed_kmph": "Prędkość [km/h]",
            "congestion_index_pct": "Indeks korka [%]"
        }
    )

    fig.update_layout(mapbox_style="carto-positron")
    fig.show()


def plot_interactive_delay_by_hour(top_n=8):
    """Rysuje interaktywny wykres liniowy średnich opóźnień wg godzin"""
    df = load_history()


    top_points = (
        df.groupby("point_name")["delay_sec"]
        .mean()
        .sort_values(ascending=False)
        .head(top_n)
        .index
    )

    filtered = df[df["point_name"].isin(top_points)]
    hourly = (
        filtered.groupby(["hour", "point_name"])["delay_sec"]
        .mean()
        .reset_index()
    )

    fig = px.line(
        hourly,
        x="hour",
        y="delay_sec",
        color="point_name",
        markers=True,
        title=f"Średnie opóźnienie w ciągu dnia (TOP {top_n} lokalizacji)",
        labels={
            "hour": "Godzina",
            "delay_sec": "Średnie opóźnienie [s]",
            "point_name": "Skrzyżowanie / Punkt"
        }
    )


    fig.update_layout(xaxis=dict(tickmode='linear', tick0=0, dtick=1))
    fig.show()


def plot_interactive_congestion_ranking(top_n=20):
    """Rysuje interaktywny ranking zakorkowania dla najnowszego pomiaru"""
    df = load_history()

    latest_time = df["timestamp"].max()
    latest = df[df["timestamp"] == latest_time].copy()


    latest = latest.sort_values("delay_sec", ascending=True).tail(top_n)

    fig = px.bar(
        latest,
        x="delay_sec",
        y="point_name",
        orientation='h',
        color="delay_sec",
        color_continuous_scale=px.colors.sequential.Reds,
        title=f"TOP {top_n} najbardziej zakorkowanych punktów (Aktualny pomiar)",
        hover_data=["current_speed_kmph", "congestion_index_pct"],
        labels={
            "delay_sec": "Opóźnienie [s]",
            "point_name": "Lokalizacja",
            "current_speed_kmph": "Prędkość [km/h]",
            "congestion_index_pct": "Korek [%]"
        }
    )
    fig.show()


def plot_delay_by_day(df, time_column='timestamp', delay_column='delay_sec'):
    df_plot = df.copy()
    df_plot[time_column] = pd.to_datetime(df_plot[time_column])
    df_plot['day_of_week'] = df_plot[time_column].dt.day_name()
    day_map = {
        'Monday': 'Poniedziałek', 'Tuesday': 'Wtorek',
        'Wednesday': 'Środa', 'Thursday': 'Czwartek',
        'Friday': 'Piątek', 'Saturday': 'Sobota', 'Sunday': 'Niedziela'
    }
    df_plot['day_of_week_pl'] = df_plot['day_of_week'].map(day_map)
    categ = ['Poniedziałek', 'Wtorek', 'Środa', 'Czwartek', 'Piątek', 'Sobota', 'Niedziela']
    df_plot['day_of_week_pl'] = pd.Categorical(df_plot['day_of_week_pl'], categories=categ, ordered=True)
    delay_by_day = df_plot.groupby('day_of_week_pl')[delay_column].mean().reset_index()

    fig, ax = plt.subplots(figsize=(10, 6))
    norm = mcolors.Normalize(vmin=delay_by_day[delay_column].min(), vmax=delay_by_day[delay_column].max())

    custom_cmap = mcolors.LinearSegmentedColormap.from_list("custom_orange", ["#66bb6a", "#ffee58", "#ff9800"])
    colors = custom_cmap(norm(delay_by_day[delay_column]))

    bars = ax.bar(delay_by_day['day_of_week_pl'], delay_by_day[delay_column], color=colors, edgecolor='black', linewidth=0.8)

    ax.set_title('Średnie opóźnienie ruchu w zależności od dnia tygodnia', fontsize=14, fontweight='bold', pad=15)
    ax.set_xlabel('Dzień tygodnia', fontsize=12)
    ax.set_ylabel('Średnie opóźnienie (sekundy)', fontsize=12)

    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.1f} s', xy=(bar.get_x() + bar.get_width() / 2, height), xytext=(0, 4), textcoords="offset points", ha='center', va='bottom', fontweight='bold', fontsize=10)

    ax.set_facecolor('#fafafa')
    fig.patch.set_facecolor('#ffffff')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    ax.set_axisbelow(True)

    plt.tight_layout()
    plt.show()

Zbieranie danych:

In [ ]:
#Dane juz zostały zebrane do pliku csv, wiec nie ma konieczności ponownego uruchamainaia
#collect_loop(interval_minutes=1, number_of_rounds=1)

Podsumowanie zebranych danych


In [ ]:
show_summary()

Wykres najniższych średnich prędkości

In [ ]:
plot_average_speed_by_point(top_n=25)

Top 10 najbardziej zakorkowanych punktów

In [ ]:
plot_current_congestion_ranking(top_n=10)

Opóżnienia w zaleźności od dnia

In [ ]:
df = pd.read_csv('traffic_history.csv')

plot_delay_by_day(df)

Heatmapa opóznień

In [ ]:
plot_delay_heatmap(top_n=25)


Średnie opóznienie według godziny





In [ ]:
plot_interactive_delay_by_hour(top_n=8)

Wyniki najbardziej zakorkowanych punktów dla najnowszego pomiaru

In [ ]:
plot_interactive_congestion_ranking(top_n=15)

Pokazanie zakorkowanych węzłów na mapie

In [ ]:
plot_interactive_map()

In [ ]:
!pip install gtfs-realtime-bindings
import osmnx as ox
import folium
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import pandas as pd
import networkx as nx
import requests
import urllib.request
import zipfile
import io
import re
from google.transit import gtfs_realtime_pb2
from IPython.display import display

def _normalize_stop_id(value):
    if pd.isna(value): return None
    s = str(value).strip()
    match = re.findall(r'(\d+)', s)
    return match[-1] if match else s

def _fetch_live_kmk_delays():
    """Pobiera aktualne opóźnienia GTFS-RT (TripUpdates) z ZTP Kraków"""
    urls = {
        "Tramwaje": "https://gtfs.ztp.krakow.pl/TripUpdates_T.pb",
        "Autobusy": "https://gtfs.ztp.krakow.pl/TripUpdates_A.pb"
    }
    rows = []
    for v_type, url in urls.items():
        try:
            response = requests.get(url, timeout=20, headers={"User-Agent": "Mozilla/5.0"})
            if not response.content: continue
            feed = gtfs_realtime_pb2.FeedMessage()
            feed.ParseFromString(response.content)

            for entity in feed.entity:
                if not entity.HasField("trip_update"): continue
                tu = entity.trip_update
                for stu in tu.stop_time_update:
                    delay = None
                    if stu.HasField("arrival"):
                        delay = stu.arrival.delay
                    elif stu.HasField("departure"):
                        delay = stu.departure.delay

                    if delay is not None:
                        rows.append({
                            "typ": v_type,
                            "stop_id": str(stu.stop_id),
                            "delay_min": float(delay) / 60.0
                        })
        except Exception as e:
            print(f"Nie udało się pobrać danych RT dla: {v_type} ({e})")
    return pd.DataFrame(rows)

def _fetch_gtfs_static_stops():
    """Pobiera statyczne lokalizacje i nazwy przystanków z plików ZIP"""
    urls = {
        "Tramwaje": "https://gtfs.ztp.krakow.pl/GTFS_KRK_T.zip",
        "Autobusy": "https://gtfs.ztp.krakow.pl/GTFS_KRK_A.zip"
    }
    all_stops = []
    for v_type, url in urls.items():
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=20) as resp:
                with zipfile.ZipFile(io.BytesIO(resp.read())) as z:
                    with z.open("stops.txt") as f:
                        s = pd.read_csv(f, dtype=str)
                        s["typ"] = v_type
                        all_stops.append(s)
        except Exception as e:
            print(f"Nie można załadować statycznego pliku GTFS dla: {v_type}")
    return pd.concat(all_stops, ignore_index=True) if all_stops else pd.DataFrame()


def plot_rich_multimodal_mobility_map():
    print("Przygotowywanie danych TomTom (Ruch samochodowy)...")
    try:
        df_tomtom = load_history()
        latest_time = df_tomtom["timestamp"].max()
        df_tomtom_latest = df_tomtom[df_tomtom["timestamp"] == latest_time].copy()
        df_tomtom_agg = df_tomtom_latest.groupby(['point_name', 'lat', 'lon'], as_index=False).agg(
            delay_min=('delay_sec', lambda x: x.mean() / 60.0),
            measurements=('timestamp', 'count')
        ).rename(columns={'point_name': 'name'})
        df_tomtom_agg['source'] = 'TomTom (Auta Osobowe)'
        print(f"   -> Pobrano {len(df_tomtom_agg)} punktów pomiarowych TomTom.")
    except Exception as e:
        print(f"Problem z ładowaniem danych historii TomTom: {e}")
        df_tomtom_agg = pd.DataFrame()

    print("Pobieranie i przetwarzanie danych KMK (Komunikacja Miejska)...")
    df_live_delays = _fetch_live_kmk_delays()
    df_static_stops = _fetch_gtfs_static_stops()

    if not df_live_delays.empty and not df_static_stops.empty:
        df_live_delays["merge_key"] = df_live_delays["stop_id"].apply(_normalize_stop_id)
        df_static_stops["merge_key"] = df_static_stops["stop_id"].apply(_normalize_stop_id)
        df_kmk_merged = df_live_delays.merge(df_static_stops, on=["merge_key", "typ"])
        df_kmk_merged["stop_lat"] = pd.to_numeric(df_kmk_merged["stop_lat"], errors="coerce")
        df_kmk_merged["stop_lon"] = pd.to_numeric(df_kmk_merged["stop_lon"], errors="coerce")
        df_kmk_merged = df_kmk_merged[df_kmk_merged['delay_min'] > 0]
        df_kmk_agg = df_kmk_merged.groupby(["stop_name", "stop_lat", "stop_lon", "typ"], as_index=False).agg(
            delay_min=("delay_min", "mean"),
            measurements=("delay_min", "count")
        ).rename(columns={'stop_name': 'name', 'stop_lat': 'lat', 'stop_lon': 'lon'})
        df_kmk_agg['source'] = 'ZTP KMK (' + df_kmk_agg['typ'] + ')'
        print(f"   -> Pobrano i dopasowano {len(df_kmk_agg)} opóźnionych przystanków autobusowych/tramwajowych.")
    else:
        df_kmk_agg = pd.DataFrame()

    df_all_mobility = pd.concat([df_tomtom_agg, df_kmk_agg], ignore_index=True).dropna(subset=['lat', 'lon', 'delay_min'])
    if df_all_mobility.empty:
        print("Brak danych (zarówno TomTom jak i KMK) do wyrenderowania mapy.")
        return

    print("Pobieranie pełnej siatki ulic Krakowa (OSMnx)...")
    lokalizacja = "Kraków, Poland"
    try:
        G = ox.graph_from_place(lokalizacja, network_type="drive", simplify=True)
    except:
        G = ox.graph_from_place(lokalizacja, network_type="drive")

    print("Budowanie szerokich korytarzy natężenia ruchu (Promień 500m)...")
    features = []
    cmap = plt.get_cmap('RdYlGn_r')
    norm = mcolors.Normalize(vmin=0, vmax=10)

    for idx, row in df_all_mobility.iterrows():
        lat, lon = row['lat'], row['lon']
        delay = row['delay_min']
        name = row['name']
        src = row['source']
        meas = row['measurements']
        try:
            nearest_node = ox.distance.nearest_nodes(G, X=lon, Y=lat)
            subgraph = nx.ego_graph(G, nearest_node, radius=500, distance='length')
            color_hex = mcolors.to_hex(cmap(norm(delay)))
            weight = 8 if 'TomTom' in src else 5
            opacity = 0.85 if 'TomTom' in src else 0.65
            for u, v, key, data in subgraph.edges(keys=True, data=True):
                if 'geometry' in data:
                    line_coords = [[x, y] for x, y in data['geometry'].coords]
                else:
                    line_coords = [[G.nodes[u]['x'], G.nodes[u]['y']], [G.nodes[v]['x'], G.nodes[v]['y']]]
                features.append({
                    "type": "Feature",
                    "geometry": {"type": "LineString", "coordinates": line_coords},
                    "properties": {
                        "style": {"color": color_hex, "weight": weight, "opacity": opacity},
                        "name": name,
                        "source": src,
                        "delay": f"{round(delay, 1)} min",
                        "measurements": int(meas)
                    }
                })
        except:
            continue

    print("Generowanie docelowej interaktywnej mapy...")
    folium_map = folium.Map(location=[50.0614, 19.9383], zoom_start=12.5, tiles="cartodbdark_matter")
    folium.GeoJson(
        {"type": "FeatureCollection", "features": features},
        style_function=lambda feature: feature["properties"]["style"],
        tooltip=folium.GeoJsonTooltip(
            fields=['name', 'source', 'delay', 'measurements'],
            aliases=['Lokalizacja/Przystanek:', 'Źródło danych:', 'Średnie opóźnienie:', 'Liczba pojazdów/pomiarów:'],
            style="background-color: #222222; color: #ffffff; font-family: sans-serif; font-size: 12px; padding: 8px; border-radius: 4px; border: 1px solid #444;"
        )
    ).add_to(folium_map)

    print("Wykres skończony!")
    display(folium_map)

plot_rich_multimodal_mobility_map()